# aqui rodamos 14 meses deepseek




In [5]:
PROMPT_TEXT = r'''
def build_prompt(
    tasks_str: str,
    clean_titles: list[str],
    domain: str,
    job_ad_title: str,
    job_sector_category: str,
    full_ad_text: str,
) -> str:
    n_candidates = len(clean_titles)
    numbered = "\n".join(f"{i+1}. {t}" for i, t in enumerate(clean_titles))

    full_excerpt = (full_ad_text or "").strip()[:700]
    full_block = (
        f"\nFULL AD EXCERPT (tools/duties evidence only):\n{full_excerpt}\n"
        if full_excerpt
        else ""
    )

    # No <think>. Hard JSON-only instruction is repeated.
    return f"""
<|im_start|>system
Return ONLY a valid JSON object with exactly one key "drop". No extra text.
<|im_end|>

<|im_start|>user
Return ONLY a valid JSON object with exactly one key "drop". No prose. No markdown.

TASK
Audit occupation matches. DROP candidates that are NOT functional matches for the job.

KEEP POLICY
There are {n_candidates} candidates (1-based).
- Default: KEEP 2–3 candidates.
- KEEP 1 ONLY if you are certain every other candidate is clearly wrong.
- If more than 3 are valid, drop the most generic ones to fit the 3-candidate cap.
- When in doubt, KEEP rather than DROP if functionally plausible.

RANKING NOTE
Candidates are ranked by cosine similarity (rank 1 = highest cosine). This is a weak prior.
Do NOT assume rank 1 is correct.

JOB CONTEXT (SOURCE OF TRUTH)
Title: {job_ad_title}
Sector: {job_sector_category}
Domain: {domain}
Tasks: {tasks_str}
{full_block}

CANDIDATES (1-based)
{numbered}

ANCHOR (FUNCTION FIRST)
- Identify the functional anchor from title + tasks (function, not seniority).
- You MUST keep the anchor unless core evidence contradicts it.
- Title keyword lock: if the title contains a clear functional keyword and a direct-match candidate exists, you MUST keep it.

GATES
- Manager rule: if title does NOT include Manager/Lead/Director, keep manager roles only if tasks mention supervision, rotas, hiring, budgeting.
- IT lock: if title/tasks mention concrete tech (Python/SQL/APIs/systems), keep relevant IT roles even if domain is non-IT.

FINAL CHECK
- Keep 2–3 by default.
- Keeping only 1 requires clear mismatch for all others.

OUTPUT
Return ONLY JSON: {{"drop":[...]}}
<|im_end|>

<|im_start|>assistant
""".strip()


def build_retry_prompt(
    tasks_str: str,
    clean_titles: list[str],
    domain: str,
    job_ad_title: str,
    job_sector_category: str,
) -> str:
    numbered = "\n".join(f"{i+1}. {t}" for i, t in enumerate(clean_titles))

    return f"""
<|im_start|>system
Return ONLY valid JSON: {{"drop":[...]}}. No extra text.
<|im_end|>

<|im_start|>user
Return ONLY valid JSON with key "drop": {{"drop":[...]}}. No prose. No markdown.

Title: {job_ad_title}
Sector: {job_sector_category}
Domain: {domain}
Tasks: {tasks_str}

Candidates (1-based):
{numbered}

Rules:
- Keep 2–3 by default.
- Keep 1 only if very sure all others are clearly wrong.
- Drop only clearly wrong functions.

Output ONLY JSON: {{"drop":[...]}}
<|im_end|>

<|im_start|>assistant
""".strip()
'''

## bge e5 and bge all in the same batch

In [1]:
import subprocess, re
from pathlib import Path

#SBATCH = Path("/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/dev/sbatch/run_deepseek_drop_bge.sbatch")
SBATCH = Path("/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/dev/sbatch/vllm_1gpu_array_bge_e5_gte.sbatch")
assert SBATCH.exists(), f"Missing sbatch file: {SBATCH}"

out = subprocess.check_output(["sbatch", str(SBATCH)], text=True)
jobid = re.search(r"(\d+)", out).group(1)
print("JOBID:", jobid)
bge_jobid = jobid

JOBID: 2431184


# monitoring

In [2]:
import time, subprocess
from pathlib import Path

LOG_DIR = Path("/projects/a5u/adu_dev/aisi-economy-index/logs")
def sh(cmd):
    return subprocess.run(cmd, shell=True, text=True, capture_output=True).stdout.strip()
while True:
    print("\n=== squeue ===")
    sq = sh(f"squeue -j {jobid}")
    print(sq if sq else "<finished>")

    out_logs = sorted(LOG_DIR.glob(f"*{jobid}*.out"))
    err_logs = sorted(LOG_DIR.glob(f"*{jobid}*.err"))

    if out_logs:
        print("\n--- STDOUT ---")
        print(sh(f"tail -n 20 {out_logs[-1]}"))

    if err_logs:
        print("\n--- STDERR ---")
        print(sh(f"tail -n 20 {err_logs[-1]}"))

    if not sq:
        break

    time.sleep(5)



=== squeue ===
JOBID         USER PARTITION                     NAME ST TIME_LIMIT       TIME  TIME_LEFT NODES NODELIST(REASON)

--- STDOUT ---
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
(EngineCore_DP0 pid=81333) INFO 02-21 15:17:22 [parallel_state.py:1208] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=81333) WARNING 02-21 15:17:22 [topk_topp_sampler.py:66] FlashInfer is not available. Falling back to the PyTorch-native implementation of top-p & top-k sampling. For the best performance, please install FlashInfer.
(EngineCore_DP0 pid=81333) INFO 02-21 15:17:22 [gpu_model_runner.py:2602] Starting to load model /projects/public/brics/cache/models--deepseek-ai--DeepSeek-R1-Distill-Qwen-32B/snapshots/711ad2ea6aa40cfca18895e8aca02ab92df1a746...
(EngineCore_DP0 pid=81333) INFO 02-21 15:17:22 [gpu_mod

In [3]:
import os
import json
from pathlib import Path

# Base directory for your production runs
BASE_DIR = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/dev/llm_negative_selection/deepseek")

# Models to check
MODELS = ["bge_large", "e5_large", "gte_large"]

def print_most_recent_outputs():
    print(f"{'MODEL':<10} | {'LATEST FILENAME'}")
    print("-" * 60)
    
    for model_dir_name in MODELS:
        # Construct path to the month folder
        target_path = BASE_DIR / model_dir_name / "adzuna_month01"
        
        if not target_path.exists():
            print(f"{model_dir_name:<10} | Path not found: {target_path}")
            continue
            
        # Get all jsonl files in the directory
        files = list(target_path.glob("*.jsonl"))
        
        if not files:
            print(f"{model_dir_name:<10} | No .jsonl files found.")
            continue
            
        # Find the most recent file based on modification time (mtime)
        latest_file = max(files, key=os.path.getmtime)
        
        print(f"{model_dir_name:<10} | {latest_file.name}")
        
        # Optional: Print the last 2 records from this specific latest file
        try:
            with open(latest_file, 'r', encoding='utf-8') as f:
                lines = f.readlines()
                if lines:
                    last_record = json.loads(lines[-1])
                    print(f"   -> Last Job ID: {last_record.get('job_id')}")
                    print(f"   -> Kept: {last_record.get('final')}")
        except Exception as e:
            print(f"   -> Error reading file: {e}")
        print("-" * 60)

if __name__ == "__main__":
    print_most_recent_outputs()

MODEL      | LATEST FILENAME
------------------------------------------------------------
bge_large  | vllm_drop_adzuna_month01_0_1000000000_job2431185_task0_rank0_20260221_151637.jsonl
   -> Last Job ID: 2767917838
   -> Kept: ['First-Line Supervisors of Retail Sales Workers', 'Logisticians', 'Sales Managers']
------------------------------------------------------------
e5_large   | vllm_drop_adzuna_month01_0_1000000000_job2431199_task14_rank0_20260221_151640.jsonl
   -> Last Job ID: 2767917838
   -> Kept: ['Sales Managers', 'Stockers and Order Fillers']
------------------------------------------------------------
gte_large  | vllm_drop_adzuna_month01_0_1000000000_job2431213_task28_rank0_20260221_151642.jsonl
   -> Last Job ID: 2767917838
   -> Kept: ['First-Line Supervisors of Retail Sales Workers', 'Stockers and Order Fillers']
------------------------------------------------------------


## Report deepseek

In [1]:
import os
os.getcwd()
import importlib
import gen_report_helper as grh # gen_report_helper.py 
from pathlib import Path
importlib.reload(grh) 

res = grh.generate_LLM_MODEL_full_report_and_plots(
    JSONL_BASE_DIR=Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/dev/llm_negative_selection/deepseek"),
    NPZ_BASE_DIR=Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/dev/llm_negative_selection"),
    llm_model="DeepSeek-R1-Distill-Qwen-32B",
    prompt_text=PROMPT_TEXT,
    reports_subdir="reports/deepseek_reports"
)
res

NameError: name 'PROMPT_TEXT' is not defined